In [1]:
import re
import pandas as pd

def parse_log_file(path):
    # Initialize lists to store the parsed data
    epochs = []
    selected_clients = []
    accuracies = []
    losses = []
    run_times = []
    log_file_path = f'{path}/output.log'  # Path to the log file
    output_csv_path = f'{path}/epoch_results.csv'  # Path to save the CSV file
    # Regular expressions to match the relevant lines
    epoch_pattern = re.compile(r'\| current_epoch (\d+) \|')
    client_select_pattern = re.compile(r'SchedulerThread (?:schedule Group \[ \d+ \], )?select\( \d+ clients\):')
    client_id_pattern = re.compile(r'(\d+) \|')
    epoch_result_pattern = re.compile(r'Epoch\(t\): (\d+) accuracy: ([\d.]+) loss ([\d.]+) run_time: ([\d.]+)')

    # Read the log file
    with open(log_file_path, 'r') as file:
        current_epoch = None
        clients = []
        for line in file:
            # Match the current epoch line
            epoch_match = epoch_pattern.search(line)
            if epoch_match:
                current_epoch = int(epoch_match.group(1))

            # Match the client selection line
            client_select_match = client_select_pattern.search(line)
            if client_select_match:
                # Reset clients list for the new epoch
                clients = []
                # Read the next line to extract client IDs
                next_line = next(file, '').strip()
                client_ids = client_id_pattern.findall(next_line)
                clients = [int(client_id) for client_id in client_ids]

            # Match the epoch result line
            epoch_result_match = epoch_result_pattern.search(line)
            if epoch_result_match:
                epoch = int(epoch_result_match.group(1))
                accuracy = float(epoch_result_match.group(2))
                loss = float(epoch_result_match.group(3))
                run_time = float(epoch_result_match.group(4))

                # Ensure the epoch and clients are matched correctly
                if current_epoch == epoch:
                    epochs.append(current_epoch)
                    selected_clients.append(clients)
                    accuracies.append(accuracy)
                    losses.append(loss)
                    run_times.append(run_time)
                else:
                    print(f"Warning: Mismatch between epoch {current_epoch} and result epoch {epoch}")

    # Create a DataFrame from the parsed data
    data = {
        'epoch': epochs,
        'selected_clients': selected_clients,
        'accuracy': accuracies,
        'loss': losses,
        'run_time': run_times
    }
    df = pd.DataFrame(data)

    # Save the DataFrame to a CSV file
    df.to_csv(output_csv_path, index=False)
    print(f"Data saved to {output_csv_path}")

import pandas as pd
import ast

def cal_round_time(x, delay_model):
    """
    计算单个轮次的模拟时间。
    :param x: 当前轮次的数据（包含 selected_clients）。
    :param delay_model: 客户端延迟模型（DataFrame）。
    :return: 当前轮次的最大客户端延迟时间。
    """
    client_ids = x['selected_clients']
    # 提取每个客户端的延迟时间
    time_per_client = []
    for client in client_ids:
        try:
            # 查询客户端的延迟时间
            time = delay_model.loc[client, 'time']
            time_per_client.append(time)
        except KeyError:
            # 如果客户端不存在于延迟模型中，跳过或记录警告
            print(f"Warning: Client {client} not found in delay model.")
            continue
    # 返回最大延迟时间
    return max(time_per_client) if time_per_client else 0

def cal_sim_round_time(delay_model_path, epoch_result_path):
    """
    计算所有轮次的模拟时间。
    :param delay_model_path: 客户端延迟模型文件路径。
    :param epoch_result_path: 轮次结果文件路径。
    :return: 包含模拟时间和累计时间的轮次结果 DataFrame。
    """
    # 读取延迟模型和轮次结果
    delay_model = pd.read_csv(delay_model_path)
    epoch_result = pd.read_csv(epoch_result_path)

    # 将 selected_clients 列从字符串转换为列表
    epoch_result['selected_clients'] = epoch_result['selected_clients'].apply(ast.literal_eval)

    # 将 delay_model 的 client_id 设置为索引，以提高查询效率
    delay_model = delay_model.set_index('client_id')

    # 计算每个轮次的模拟时间
    epoch_result['sim_round_time'] = epoch_result.apply(lambda x: cal_round_time(x, delay_model), axis=1)

    # 计算累计时间
    epoch_result['epoch_cum_time'] = epoch_result['sim_round_time'].cumsum()

    return epoch_result

def process_and_save_results(delay_model_path, epoch_result_path, output_csv_path):
    """
    封装函数：计算模拟时间和累计时间，并保存结果为 CSV 文件。
    :param delay_model_path: 客户端延迟模型文件路径。
    :param fedavg_epoch_result: 轮次结果文件路径。
    :param output_csv_path: 输出 CSV 文件路径。
    """
    # 计算模拟时间和累计时间
    epoch_result = cal_sim_round_time(delay_model_path, epoch_result_path)

    # 保存结果为 CSV 文件
    epoch_result.to_csv(output_csv_path, index=False)
    print(f"Results saved to {output_csv_path}")

# # Example usage
# log_file_path = 'output.log'  # Path to the log file
# output_csv_path = 'epoch_results.csv'  # Path to save the CSV file
# parse_log_file(log_file_path, output_csv_path)


#### Exp2025-FashionMNIST-Dir0.5

In [2]:
fedavg_path = '/home/ypguo/async-FL/src/results/Exp2025-FashionMNIST-Dir0.5/fedavg-cnn-dir0.5'
fedlc_path = '/home/ypguo/async-FL/src/results/Exp2025-FashionMNIST-Dir0.5/fedlc-cnn-dir0.5'
feddocs_path = '/home/ypguo/async-FL/src/results/Exp2025-FashionMNIST-Dir0.5/FedDocs-CNN-dir0.5'
pyramidfl_path = '/home/ypguo/async-FL/src/results/Exp2025-FashionMNIST-Dir0.5/PyramidFL-cnn-dir0.5'
baseline_path_list = [fedavg_path,fedlc_path,feddocs_path,pyramidfl_path]
for path in baseline_path_list:
    parse_log_file(path)
    print("finished " + path.split("/")[-1])




Data saved to /home/ypguo/async-FL/src/results/Exp2025-FashionMNIST-Dir0.5/fedavg-cnn-dir0.5/epoch_results.csv
finished fedavg-cnn-dir0.5
Data saved to /home/ypguo/async-FL/src/results/Exp2025-FashionMNIST-Dir0.5/fedlc-cnn-dir0.5/epoch_results.csv
finished fedlc-cnn-dir0.5
Data saved to /home/ypguo/async-FL/src/results/Exp2025-FashionMNIST-Dir0.5/FedDocs-CNN-dir0.5/epoch_results.csv
finished FedDocs-CNN-dir0.5
Data saved to /home/ypguo/async-FL/src/results/Exp2025-FashionMNIST-Dir0.5/PyramidFL-cnn-dir0.5/epoch_results.csv
finished PyramidFL-cnn-dir0.5


In [3]:
# 示例调用
delay_model_path = '/home/ypguo/async-FL/src/clients_info/clients_info_dir0.5_cnn_delaymodel.csv'
# fedavg_epoch_result = '/home/ypguo/async-FL/src/results/Exp2025-FashionMNIST-Dir0.5/fedavg-cnn-dir0.5/epoch_results.csv'
for path in baseline_path_list:
    epoch_result_path = path + '/epoch_results.csv'
    output_csv_path = path + '/epoch_results_with_cumtime.csv'
    process_and_save_results(delay_model_path, epoch_result_path, output_csv_path)
    print("finished " + path.split("/")[-1])


Results saved to /home/ypguo/async-FL/src/results/Exp2025-FashionMNIST-Dir0.5/fedavg-cnn-dir0.5/epoch_results_with_cumtime.csv
finished fedavg-cnn-dir0.5
Results saved to /home/ypguo/async-FL/src/results/Exp2025-FashionMNIST-Dir0.5/fedlc-cnn-dir0.5/epoch_results_with_cumtime.csv
finished fedlc-cnn-dir0.5
Results saved to /home/ypguo/async-FL/src/results/Exp2025-FashionMNIST-Dir0.5/FedDocs-CNN-dir0.5/epoch_results_with_cumtime.csv
finished FedDocs-CNN-dir0.5
Results saved to /home/ypguo/async-FL/src/results/Exp2025-FashionMNIST-Dir0.5/PyramidFL-cnn-dir0.5/epoch_results_with_cumtime.csv
finished PyramidFL-cnn-dir0.5


In [4]:
import pandas as pd
import numpy as np
import os

def analyze_federated_learning_csv(csv_path, accuracy_thresholds, total_rounds):
    """
    分析联邦学习CSV文件以提取性能指标。
    
    参数:
        csv_path (str): 包含轮次结果的CSV文件路径
        accuracy_thresholds (list): 需要查找的准确率阈值列表 (例如, [75, 80, 85])
        total_rounds (int): 要考虑的总轮次数
        
    返回:
        dict: 包含性能指标的字典:
            - epoch_at_threshold: 阈值到首次达到该阈值的轮次的映射
            - time_at_threshold: 阈值到首次达到该阈值的时间的映射（分钟）
            - total_time: 指定总轮次的时间（分钟）
            - best_accuracy: 达到的最佳准确率
    """
    # 读取CSV文件
    df = pd.read_csv(csv_path)
    
    # 初始化结果
    results = {
        'epoch_at_threshold': {},
        'time_at_threshold': {},
        'total_time': None,
        'best_accuracy': None
    }
    
    # 限制数据范围到total_rounds
    df_limited = df[df['epoch'] <= total_rounds]
    
    # 找到达到的最佳准确率（仅考虑total_rounds内的数据）
    if not df_limited.empty:
        results['best_accuracy'] = df_limited['accuracy'].max()
    else:
        results['best_accuracy'] = 0
    
    # 找到首次达到每个准确率阈值的时间
    for threshold in accuracy_thresholds:
        threshold_rows = df[df['accuracy'] >= threshold]
        
        if not threshold_rows.empty:
            first_row = threshold_rows.iloc[0]
            results['epoch_at_threshold'][threshold] = int(first_row['epoch'])
            # 将秒转换为分钟
            results['time_at_threshold'][threshold] = float(first_row['epoch_cum_time']) / 60.0
        else:
            # 如果未达到阈值，则设置为None
            results['epoch_at_threshold'][threshold] = None
            results['time_at_threshold'][threshold] = None
    
    # 获取指定总轮次的时间（转换为分钟）
    if total_rounds <= df['epoch'].max():
        total_rounds_row = df[df['epoch'] == total_rounds]
        if not total_rounds_row.empty:
            results['total_time'] = float(total_rounds_row.iloc[0]['epoch_cum_time']) / 60.0
        else:
            # 如果找不到确切的轮次但在范围内，使用最后一个可用的轮次
            results['total_time'] = float(df['epoch_cum_time'].iloc[-1]) / 60.0
    else:
        # 如果total_rounds超出数据范围，使用最后一个可用的轮次
        results['total_time'] = float(df['epoch_cum_time'].iloc[-1]) / 60.0
    
    return results

def format_as_row(method, distribution, results, accuracy_thresholds):
    """
    将结果格式化为表格的单行数据。
    
    参数:
        method (str): 方法名称 (例如 "FedAvg")
        distribution (str): 分布类型 (例如 "IID" 或 "nonIID(0.5)")
        results (dict): analyze_federated_learning_csv的结果
        accuracy_thresholds (list): 准确率阈值列表
        
    返回:
        list: 格式化的数据行，类似图片中的表格
    """
    # 确保分布信息正确显示
    if distribution.lower() == "iid":
        row = ["IID", method]
    else:
        # 确保nonIID格式正确
        if not distribution.lower().startswith("noniid"):
            distribution = f"nonIID({distribution})"
        row = [distribution, method]
    
    # 为每个阈值添加轮次和时间
    for threshold in accuracy_thresholds:
        epoch = results['epoch_at_threshold'].get(threshold)
        time = results['time_at_threshold'].get(threshold)
        
        if epoch is not None and time is not None:
            row.append(epoch)
            row.append(round(time, 2))  # 时间已在analyze_federated_learning_csv中转换为分钟
        else:
            # 如果未达到准确率，则设置为0而不是"-"
            row.append(0)
            row.append(0)
    
    # 添加总时间和最佳准确率
    row.append(round(results['total_time'], 2))  # 时间已在analyze_federated_learning_csv中转换为分钟
    row.append(round(results['best_accuracy'], 2))
    
    return row

def analyze_multiple_algorithms(csv_paths_dict, accuracy_thresholds, total_rounds, output_excel_path):
    """
    分析多个算法在不同数据分布上的性能，并生成Excel文件。
    
    参数:
        csv_paths_dict (dict): 字典结构为 {分布: {算法: csv路径, ...}, ...}
        accuracy_thresholds (list): 准确率阈值列表 (例如 [75, 80, 85])
        total_rounds (int): 要考虑的总轮次数
        output_excel_path (str): 输出Excel文件的路径
        
    返回:
        DataFrame: 包含所有结果的DataFrame
    """
    # 创建标题
    header = ["数据集", "方法"]
    for threshold in accuracy_thresholds:
        header.extend([f"{threshold}%轮次", f"{threshold}%时间(分钟)"])
    header.extend(["总时间(分钟)", "最佳准确率"])
    
    # 收集所有行
    all_rows = []
    
    # 遍历每个分布和算法
    for distribution, algorithms in csv_paths_dict.items():
        for method, csv_path in algorithms.items():
            if not os.path.exists(csv_path):
                print(f"警告: CSV文件 {csv_path} 未找到。")
                continue
            
            # 分析CSV
            results = analyze_federated_learning_csv(csv_path, accuracy_thresholds, total_rounds)
            
            # 格式化为行
            row = format_as_row(method, distribution, results, accuracy_thresholds)
            all_rows.append(row)
    
    # 创建DataFrame
    df = pd.DataFrame(all_rows, columns=header)
    
    # 保存到Excel
    df.to_excel(output_excel_path, index=False)
    
    print(f"结果已保存到 {output_excel_path}")
    
    return df

# 使用示例
if __name__ == "__main__":
    # 准确率阈值列表
    accuracy_thresholds = [75, 80, 85]
    # 总轮次
    total_rounds = 500
    # 输出Excel文件路径
    output_excel_path = "federated_learning_results.xlsx"

    # 示例数据路径
    fedavg_path = 'src/results/Exp2025-FashionMNIST-Dir0.5/fedavg-cnn-dir0.5'
    fedlc_path = 'src/results/Exp2025-FashionMNIST-Dir0.5/fedlc-cnn-dir0.5'
    feddocs_path = 'src/results/Exp2025-FashionMNIST-Dir0.5/FedDocs-CNN-dir0.5'
    pyramidfl_path = 'src/results/Exp2025-FashionMNIST-Dir0.5/PyramidFL-cnn-dir0.5'
    
    baseline_path_list = [fedavg_path, fedlc_path, feddocs_path, pyramidfl_path]
    csv_paths_dict = {}
    
    for path in baseline_path_list:
        algo = path.split("/")[-1]
        algo_name = algo.split("-")[0]
        distribution = algo.split("-")[-1]
        
        # 确保正确的分布格式
        if distribution.lower() == "iid":
            dist_key = "IID"
        else:
            # 提取dir后的数字作为nonIID系数
            if distribution.lower().startswith("dir"):
                coef = distribution[3:]  # 提取dir后的数字
                dist_key = f"nonIID({coef})"
            else:
                dist_key = f"nonIID({distribution})"
        
        # 初始化字典项
        if dist_key not in csv_paths_dict:
            csv_paths_dict[dist_key] = {}
        
        # 添加算法路径
        csv_paths_dict[dist_key][algo_name] = f"{path}/epoch_results_with_cumtime.csv"
    
    # 分析并生成Excel
    results_df = analyze_multiple_algorithms(
        csv_paths_dict=csv_paths_dict,
        accuracy_thresholds=accuracy_thresholds,
        total_rounds=total_rounds,
        output_excel_path=output_excel_path
    )
    
    # 显示结果
    print(results_df)

    
    # 显示结果
    results_df.to_csv('/home/ypguo/async-FL/工作一对比结果/dir0.5baseline.csv',index=False)
    results_df.to_excel('/home/ypguo/async-FL/工作一对比结果/dir0.5baseline.xlsx',index=False)

结果已保存到 federated_learning_results.xlsx
           数据集         方法  75%轮次  75%时间(分钟)  80%轮次  80%时间(分钟)  85%轮次  \
0  nonIID(0.5)     fedavg     62      11.29    135      24.54    290   
1  nonIID(0.5)      fedlc     43       7.80    106      19.33    225   
2  nonIID(0.5)    FedDocs     40       7.27     90      16.54    185   
3  nonIID(0.5)  PyramidFL     30       5.28     80      14.31    158   

   85%时间(分钟)  总时间(分钟)  最佳准确率  
0      53.24    91.69  86.87  
1      41.17    91.69  88.09  
2      33.92    91.81  88.31  
3      28.74    92.41  88.99  
